# DATA 2x01 Group Assignment 2026

**Topic:** NSW regional statistics, Greater Sydney POI data, SA2 resource scoring, and report analysis.

**Due:** 18 May 2026, 11:59 PM

This notebook is structured to document the full workflow required for the group submission.

## Deliverables Checklist

- PDF report
- Jupyter Notebook describing the full workflow
- Tutor conversation in Week 12 or Week 13
- One group ZIP file submitted to Canvas

## Setup

In [33]:
from pathlib import Path
from tabulate import tabulate
import json
import math
import sqlite3
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd

DATA_DIR = Path.cwd()
CSV_PATH = DATA_DIR / "Region summary_ New South Wales STE 1.csv"
DB_PATH = DATA_DIR / "data2001_assignment.sqlite"

CSV_PATH

WindowsPath('c:/Users/micro/OneDrive/Documents/DATA2001/DATA2001_assignment/Region summary_ New South Wales STE 1.csv')

# Task 1: NSW Summary Statistics

## 1.1 Load the NSW Region Summary CSV

In [34]:
df_raw = pd.read_csv(CSV_PATH)
df_raw.head()

,Measure Code,Parent Description,Description,2011,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,ERP_P_20,Estimated resident population - year ended 30 ...,Estimated resident population (no.),NaN,NaN,NaN,NaN,NaN,8046748.0,8110610.0,8097062.0,8166704.0,8341199.0,8479314.0,NaN
1,ERP_21,Estimated resident population - year ended 30 ...,Population density (persons/km2),NaN,NaN,NaN,NaN,NaN,10.0,10.1,10.1,10.2,10.4,10.6,NaN
2,ERP_M_20,Estimated resident population - year ended 30 ...,Estimated resident population - males (no.),NaN,NaN,NaN,NaN,NaN,3999452.0,4030710.0,4025393.0,4059763.0,4149032.0,4217861.0,NaN
3,ERP_F_20,Estimated resident population - year ended 30 ...,Estimated resident population - females (no.),NaN,NaN,NaN,NaN,NaN,4047296.0,4079900.0,4071669.0,4106941.0,4192167.0,4261453.0,NaN
4,ERP_19,Estimated resident population - year ended 30 ...,Median age - males (years),NaN,NaN,NaN,NaN,NaN,36.8,37.2,37.7,37.7,37.5,37.5,NaN


In [35]:
print("Shape:", df_raw.shape)
display(df_raw.info())
display(df_raw.describe(include="all"))

Shape: (800, 15)
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Measure Code        800 non-null    str    
 1   Parent Description  800 non-null    str    
 2   Description         800 non-null    str    
 3   2011                259 non-null    float64
 4   2015                20 non-null     float64
 5   2016                457 non-null    float64
 6   2017                48 non-null     float64
 7   2018                125 non-null    float64
 8   2019                270 non-null    float64
 9   2020                291 non-null    float64
 10  2021                692 non-null    float64
 11  2022                293 non-null    float64
 12  2023                207 non-null    float64
 13  2024                209 non-null    float64
 14  2025                3 non-null      float64
dtypes: float64(12), str(3)
memory usage: 93.9 KB


None

,Measure Code,Parent Description,Description,2011,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
count,800,800,800,2.590000e+02,2.000000e+01,4.570000e+02,4.800000e+01,1.250000e+02,2.700000e+02,2.910000e+02,6.920000e+02,2.930000e+02,2.070000e+02,2.090000e+02,3.000000
unique,800,104,751,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,ERP_P_20,Estimated resident population - Males - year e...,Employed (no.),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,36,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,6.735284e+05,8.022549e+06,1.530573e+06,4.820513e+06,2.249978e+06,1.133374e+06,1.103433e+06,6.610403e+05,3.916004e+05,2.247894e+05,3.035678e+05,8271.666667
std,NaN,NaN,NaN,4.126540e+06,1.919068e+07,7.971771e+06,1.495873e+07,9.359767e+06,6.384606e+06,6.072784e+06,4.611798e+06,1.148907e+06,8.011850e+05,1.048662e+06,8775.542395
min,NaN,NaN,NaN,1.000000e-01,2.500000e+01,3.000000e-01,5.200000e+00,1.100000e+00,1.200000e+00,1.200000e+00,-1.682200e+04,-3.933500e+04,-3.415800e+04,-3.086500e+04,790.000000
25%,NaN,NaN,NaN,7.950000e+00,5.945000e+01,8.700000e+00,1.094500e+03,1.125000e+03,3.815000e+01,5.250000e+01,7.875000e+00,6.000000e+01,7.200000e+00,7.200000e+00,3442.000000
50%,NaN,NaN,NaN,5.990000e+01,4.079050e+04,6.090000e+01,1.431480e+04,4.201200e+04,3.605600e+04,3.649400e+04,2.956450e+03,4.103200e+04,3.746600e+04,3.182700e+04,6094.000000
75%,NaN,NaN,NaN,6.621400e+04,3.001438e+06,1.271790e+05,6.921535e+05,3.736930e+05,2.556548e+05,2.614125e+05,1.513840e+05,2.498940e+05,2.328595e+05,2.409830e+05,12012.500000


## 1.2 Data Cleaning

In [36]:
df_clean = df_raw.copy()

# Standardise column names and text fields.
df_clean.columns = df_clean.columns.str.strip()
text_columns = ["Measure Code", "Parent Description", "Description"]
for column in text_columns:
    df_clean[column] = df_clean[column].astype("string").str.strip()

year_columns = [column for column in df_clean.columns if column.isdigit()]
df_clean[year_columns] = df_clean[year_columns].apply(pd.to_numeric, errors="coerce")

# Remove exact duplicate rows if present.
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

print("Cleaned shape:", df_clean.shape)
df_clean.head()

Cleaned shape: (800, 15)


,Measure Code,Parent Description,Description,2011,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,ERP_P_20,Estimated resident population - year ended 30 ...,Estimated resident population (no.),NaN,NaN,NaN,NaN,NaN,8046748.0,8110610.0,8097062.0,8166704.0,8341199.0,8479314.0,NaN
1,ERP_21,Estimated resident population - year ended 30 ...,Population density (persons/km2),NaN,NaN,NaN,NaN,NaN,10.0,10.1,10.1,10.2,10.4,10.6,NaN
2,ERP_M_20,Estimated resident population - year ended 30 ...,Estimated resident population - males (no.),NaN,NaN,NaN,NaN,NaN,3999452.0,4030710.0,4025393.0,4059763.0,4149032.0,4217861.0,NaN
3,ERP_F_20,Estimated resident population - year ended 30 ...,Estimated resident population - females (no.),NaN,NaN,NaN,NaN,NaN,4047296.0,4079900.0,4071669.0,4106941.0,4192167.0,4261453.0,NaN
4,ERP_19,Estimated resident population - year ended 30 ...,Median age - males (years),NaN,NaN,NaN,NaN,NaN,36.8,37.2,37.7,37.7,37.5,37.5,NaN


In [37]:
missing_summary = (
    df_clean.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(df_clean) * 100).round(2)
missing_summary

,missing_count,missing_percent
Measure Code,0,0.00
Parent Description,0,0.00
Description,0,0.00
2011,541,67.62
2015,780,97.50
2016,343,42.88
2017,752,94.00
2018,675,84.38
2019,530,66.25
2020,509,63.62


### Cleaning Notes

Record the cleaning decisions here:

- Which columns were used as identifiers?
- Which years contain usable values?
- Were missing values meaningful, unavailable, or errors?
- Were duplicate rows found?
- Did any values require type conversion?

## 1.3 Derived Statistics

Each group member should contribute 5 derived statistics. Use this section to clearly label each member's work.

In [38]:
latest_year = max(year_columns, key=int)
usable_years = [column for column in year_columns if df_clean[column].notna().any()]
latest_usable_year = max(usable_years, key=int)

print("All year columns:", year_columns)
print("Latest year column:", latest_year)
print("Latest usable year:", latest_usable_year)

All year columns: ['2011', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Latest year column: 2025
Latest usable year: 2025


### Ronnie Luo: Derived Statistics

Planned statistics:

1. Statistic 1
2. Statistic 2
3. Statistic 3
4. Statistic 4
5. Statistic 5

In [39]:
# Example starting points for Task 1 statistics.
population_rows = df_clean[df_clean["Description"].str.contains("population", case=False, na=False)]
largest_latest_values = (
    df_clean[["Parent Description", "Description", latest_usable_year]]
    .dropna(subset=[latest_usable_year])
    .sort_values(latest_usable_year, ascending=False)
    .head(10)
)

display(population_rows.head(10))
display(largest_latest_values)

,Measure Code,Parent Description,Description,2011,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,ERP_P_20,Estimated resident population - year ended 30 ...,Estimated resident population (no.),NaN,NaN,NaN,NaN,NaN,8046748.0,8110610.0,8097062.0,8166704.0,8341199.0,8479314.0,NaN
1,ERP_21,Estimated resident population - year ended 30 ...,Population density (persons/km2),NaN,NaN,NaN,NaN,NaN,10.0,10.1,10.1,10.2,10.4,10.6,NaN
2,ERP_M_20,Estimated resident population - year ended 30 ...,Estimated resident population - males (no.),NaN,NaN,NaN,NaN,NaN,3999452.0,4030710.0,4025393.0,4059763.0,4149032.0,4217861.0,NaN
3,ERP_F_20,Estimated resident population - year ended 30 ...,Estimated resident population - females (no.),NaN,NaN,NaN,NaN,NaN,4047296.0,4079900.0,4071669.0,4106941.0,4192167.0,4261453.0,NaN
7,ERP_18,Estimated resident population - year ended 30 ...,Working age population (aged 15-64 years) (no.),NaN,NaN,NaN,NaN,NaN,5240576.0,5255515.0,5209700.0,5247219.0,5388091.0,5490199.0,NaN
8,ERP_17,Estimated resident population - year ended 30 ...,Working age population (aged 15-64 years) (%),NaN,NaN,NaN,NaN,NaN,65.1,64.8,64.3,64.3,64.6,64.7,NaN
163,ING_WA1,Estimated resident Aboriginal and Torres Strai...,Working age population (aged 15-64 years) (no.),124932.0,NaN,160431.0,NaN,NaN,NaN,NaN,204377.0,NaN,NaN,NaN,NaN
164,ING_WA2,Estimated resident Aboriginal and Torres Strai...,Working age population (aged 15-64 years) (%),59.9,NaN,60.4,NaN,NaN,NaN,NaN,60.2,NaN,NaN,NaN,NaN
165,ING_ERP_P,Estimated resident Aboriginal and Torres Strai...,Estimated resident Aboriginal and Torres Strai...,208476.0,NaN,265685.0,NaN,NaN,NaN,NaN,339710.0,NaN,NaN,NaN,NaN
202,ING_1564,Labour force status - Census,Total responding population aged 15-64 years (...,98901.0,NaN,127179.0,NaN,NaN,NaN,NaN,163828.0,NaN,NaN,NaN,NaN


,Parent Description,Description,2025
409,Selected Government pensions and allowances - ...,Service pension - Department of Veterans' Affa...,17931.0
410,Selected Government pensions and allowances - ...,Income support supplement - Department of Vete...,6094.0
408,Selected Government pensions and allowances - ...,Age pension - Department of Veterans' Affairs ...,790.0


### Spencer: Derived Statistics

Planned statistics:

1. Business Exit Rate
2. Employee Income Mean-Median Ratio
3. Statistic 3
4. Statistic 4
5. Statistic 5

In [50]:
rows = []

for i in range(2021, 2025):
    totals = df_clean[df_clean['Description'] == 'Total number of businesses'][str(i)].values[0]
    exits = df_clean[df_clean['Description'] == 'Total number of business exits'][str(i)].values[0]

    exit_rate = (exits / totals) * 100
    rows.append([i, f"{int(totals)}", f"{int(exits)}", f"{round(exit_rate, 1)}%"])


print(tabulate(rows, headers=["Year", "Total Businesses", "Exits", "Exit Rate"], tablefmt="pretty"))

+------+------------------+--------+-----------+
| Year | Total Businesses | Exits  | Exit Rate |
+------+------------------+--------+-----------+
| 2021 |      817603      | 96444  |   11.8%   |
| 2022 |      854727      | 105636 |   12.4%   |
| 2023 |      870951      | 128785 |   14.8%   |
| 2024 |      896560      | 122778 |   13.7%   |
+------+------------------+--------+-----------+


In [59]:
rows = []

for i in range(2018, 2023):
    median = df_clean[df_clean['Description'] == 'Median employee income ($)'][str(i)].values[0]
    mean = df_clean[df_clean['Description'] == 'Mean employee income ($)'][str(i)].values[0]
    
    ratio = mean / median
    rows.append([i, f"{int(mean)}", f"{int(median)}", f"{round(ratio, 2)}"])

print(tabulate(rows, headers=["Year", "Mean Employee Income ($)", "Median Employee Income ($)", "Mean-Median Ratio"], tablefmt="pretty"))

+------+--------------------------+----------------------------+-------------------+
| Year | Mean Employee Income ($) | Median Employee Income ($) | Mean-Median Ratio |
+------+--------------------------+----------------------------+-------------------+
| 2018 |          64509           |           51411            |       1.25        |
| 2019 |          66377           |           53104            |       1.25        |
| 2020 |          68690           |           54989            |       1.25        |
| 2021 |          71848           |           57891            |       1.24        |
| 2022 |          74232           |           58972            |       1.26        |
+------+--------------------------+----------------------------+-------------------+


### Arya: Derived Statistics


Planned statistics:

1. Statistic 1
2. Statistic 2
3. Statistic 3
4. Statistic 4
5. Statistic 5

### Ananya: Derived Statistics


Planned statistics:

1. Statistic 1
2. Statistic 2
3. Statistic 3
4. Statistic 4
5. Statistic 5

### Additional Members

Copy the Member 1 structure for each group member. Each member should add 5 derived statistics and a short explanation of why each statistic is useful.

# Task 2: Greater Sydney SA2/SA4 Points of Interest Dataset

## 2.1 Select SA4 Zones

Each group member should select one distinct Greater Sydney SA4 zone.

Record the selected zones here:

- Member 1: SA4 zone name
- Member 2: SA4 zone name
- Member 3: SA4 zone name
- Member 4: SA4 zone name

In [8]:
SELECTED_SA4_ZONES = [
    # "Sydney - City and Inner South",
    # "Parramatta",
]

# Add or load SA2 boundary data here when available.
# Expected columns: sa4_name, sa2_name, min_lon, min_lat, max_lon, max_lat
sa2_boundaries = pd.DataFrame(columns=["sa4_name", "sa2_name", "min_lon", "min_lat", "max_lon", "max_lat"])
sa2_boundaries

,sa4_name,sa2_name,min_lon,min_lat,max_lon,max_lat


## 2.2 NSW Points of Interest API Function

In [9]:
def fetch_pois_in_bbox(min_lon, min_lat, max_lon, max_lat, limit=1000):
    """Return POI records inside a bounding box.

    Update API_URL and params after confirming the NSW POI API endpoint from the Week 8 tutorial.
    """
    API_URL = "TODO_ADD_NSW_POI_API_ENDPOINT"
    params = {
        "min_lon": min_lon,
        "min_lat": min_lat,
        "max_lon": max_lon,
        "max_lat": max_lat,
        "limit": limit,
    }

    if API_URL.startswith("TODO"):
        return pd.DataFrame()

    url = API_URL + "?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url) as response:
        payload = json.loads(response.read().decode("utf-8"))

    records = payload.get("features", payload if isinstance(payload, list) else [])
    return pd.json_normalize(records)


## 2.3 Loop Through SA2 Regions

In [10]:
poi_frames = []

for _, row in sa2_boundaries.iterrows():
    if SELECTED_SA4_ZONES and row["sa4_name"] not in SELECTED_SA4_ZONES:
        continue

    pois = fetch_pois_in_bbox(row["min_lon"], row["min_lat"], row["max_lon"], row["max_lat"])
    if pois.empty:
        continue

    pois["sa4_name"] = row["sa4_name"]
    pois["sa2_name"] = row["sa2_name"]
    poi_frames.append(pois)

pois_df = pd.concat(poi_frames, ignore_index=True) if poi_frames else pd.DataFrame()
print("POI rows collected:", len(pois_df))
pois_df.head()

POI rows collected: 0


""


## 2.4 Store POI Dataset in Local Database

In [11]:
with sqlite3.connect(DB_PATH) as conn:
    if not pois_df.empty:
        pois_df.to_sql("points_of_interest", conn, if_exists="replace", index=False)
        print("Saved points_of_interest table to", DB_PATH)
    else:
        print("No POI data saved yet. Complete the API endpoint and SA2 boundary inputs first.")

No POI data saved yet. Complete the API endpoint and SA2 boundary inputs first.


# Task 3: SA2 Well-Resourced Score

In [12]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def z_score(series):
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(0, index=series.index)
    return (series - series.mean()) / std

if not pois_df.empty and "sa2_name" in pois_df.columns:
    score_df = pois_df.groupby("sa2_name").size().rename("poi_count").reset_index()
    score_df["z_poi"] = z_score(score_df["poi_count"])
    score_df["score"] = sigmoid(score_df["z_poi"])
else:
    score_df = pd.DataFrame(columns=["sa2_name", "poi_count", "z_poi", "score"])

score_df.head()

,sa2_name,poi_count,z_poi,score


## Scoring Explanation Notes

Use this section to explain:

- Why POI count is a reasonable proxy for resource availability
- Why z-score normalisation is used
- Why sigmoid scaling is used
- Whether population below 100 was excluded
- Any extensions to the formula, such as population adjustment or POI category weighting

# Task 4: Report Analysis and Visualisation

## 4.1 Key Findings from Task 1

Write the main statistical findings here after completing the derived statistics.

Possible angles:

- Population change over time
- Age structure
- Gender differences
- Density or growth indicators
- Measures with unusual changes or missingness

## 4.2 Score Visualisation Plan

Add plots here once `score_df` is populated.

Recommended visuals:

- Histogram of SA2 scores
- Top and bottom ranked SA2 regions
- Map overlay or choropleth if boundary geometry is available
- POI category breakdown by SA4 or SA2

In [13]:
if not score_df.empty:
    display(score_df.sort_values("score", ascending=False).head(10))
    display(score_df.sort_values("score", ascending=True).head(10))
    display(score_df["score"].describe())
else:
    print("Score table is empty. Complete Task 2 before generating score summaries.")

Score table is empty. Complete Task 2 before generating score summaries.


## 4.3 Limitations

Discuss the limitations of the analysis here.

Possible limitations:

- POI count does not measure service quality or capacity
- Larger SA2s may naturally contain more POIs
- Population size may need to be considered
- API completeness and category definitions may affect results
- Bounding boxes can include POIs outside the actual SA2 polygon unless geometry filtering is added

# Next Steps

1. Add group member names and selected SA4 zones.
2. Complete Task 1 derived statistics.
3. Add SA2 boundary data for selected SA4 zones.
4. Confirm the NSW Points of Interest API endpoint from the Week 8 tutorial.
5. Store POI data in the local SQLite database.
6. Generate score visualisations and write report findings.